# Digit fingerprints

Some findings are about the process that recorded the values
rather than the values themselves. Amounts typed by people pile
onto round numbers; a form that defaults one digit produces it
too often in the last position; fabricated figures rarely
reproduce the first-digit distribution of genuine measurements.
The `digits` detector runs three such checks and emits `formats`
findings: `heaping`, `digit_preference`, and `benford_deviation`.


## Heaping

For integer-valued columns with at least 500 values and at least
20 distinct values, the detector tests units 1000, 100, and 10,
largest first, and only when the unit is small against the
column's range. The measurement is the share of nonzero values
divisible by the unit, against the 1/U share a smooth recording
implies. A pile is reported when the share is at least 3× the
baseline at z ≥ 6. Zeros are excluded from the multiples test —
0 is divisible by every unit — and their share is stated in the
evidence separately, alongside the terminal-digit shares for 0
and 5.


## Terminal-digit preference

On the same eligibility, the last-digit distribution is tested
against uniformity: the chi-square statistic on 9 degrees of
freedom must clear 44, and the most frequent digit must hold at
least 15% of values. When heaping already fired on the column,
no separate digit finding is emitted: rounding to multiples of
10 produces a terminal 0 pile-up by itself, and the report keeps
one finding for the one phenomenon.


## Benford deviation

The first significant digits of positive values are compared
with Benford's law, behind an eligibility rule that coverage
records as a count of eligible columns: at least 500 values, at
least 95% of them positive, a max/min ratio of at least 1000, at
least 100 distinct values, and — for integer columns only — a
distinct share under 50%, which keeps identifier-like integer
columns out. A deviation is reported only when it is both
significant (chi-square on 8 degrees of freedom at least 34) and
material (mean absolute deviation at least 0.012). Either bar
alone would misfire: significance without materiality fires on
any large nearly-conforming sample, and materiality without
significance fires on small ones.


In [1]:
import random

import polars as pl

import onceover


def measured(seed: int, round_share: float = 0.0) -> list[int]:
    rng = random.Random(seed)
    vals = []
    for _ in range(4000):
        v = round(rng.gauss(500, 80))
        if rng.random() < round_share:
            v = round(v / 10) * 10
        vals.append(v)
    return vals


def terminal_seven(seed: int) -> list[int]:
    rng = random.Random(seed)
    vals = []
    for _ in range(4000):
        v = round(rng.gauss(500, 80))
        if rng.random() < 0.3:
            v = v - (v % 10) + 7
        vals.append(v)
    return vals


def uniform_first_digit(seed: int) -> list[float]:
    rng = random.Random(seed)
    return [rng.uniform(1, 10) * 10 ** rng.randrange(0, 4) for _ in range(4000)]


frames = {
    "rounded amounts": pl.DataFrame({"value": measured(21, round_share=0.3)}),
    "terminal 7s": pl.DataFrame({"value": terminal_seven(23)}),
    "uniform first digits": pl.DataFrame({"amount": uniform_first_digit(25)}),
}
for label, df in frames.items():
    report = onceover.profile(df)
    hits = [f for f in report.findings
            if f.get("type") in ("heaping", "digit_preference", "benford_deviation")]
    print(f"{label}: {[f['type'] for f in hits]}")
    for f in hits:
        print("  ", f["claim"])
        print("   evidence:", f["evidence"][0]["metrics"])


rounded amounts: ['heaping']
   column value heaps on multiples of 10: 36.5% of nonzero values are divisible by 10 against the 10.0% a smooth recording implies (z = 56.0); zeros are 0.0% of the column and are counted separately
   evidence: {'digit_five_share': 0.076, 'digit_zero_share': 0.3655, 'method': 'share of nonzero values divisible by the unit vs 1/unit under a smooth recording; largest firing unit reported; zeros excluded from the multiples test and reported separately', 'share_baseline': 0.1, 'share_divisible': 0.3655, 'unit': 10, 'z': 55.97231458498031, 'zero_share': 0.0}
terminal 7s: ['digit_preference']
   column value has a terminal-digit preference: digit 7 ends 37.0% of values against 10% under uniformity (chi-square 3261.2 on 9 df, bar 44)
   evidence: {'chi_square': 3261.2250000000004, 'digit_shares': [0.0805, 0.07025, 0.06475, 0.0675, 0.075, 0.06175, 0.07425, 0.37025, 0.0755, 0.06025], 'method': 'last-digit distribution over all values vs uniform; chi-square on 9 df'

## Reading the evidence

A `heaping` finding's evidence carries `unit` (the multiple the
values pile on), `share_divisible` (the share of nonzero values
divisible by it, against the 1/U baseline), and `zero_share`. A
`digit_preference` finding carries `top_digit` and
`top_digit_share` with the full last-digit distribution behind
them. A `benford_deviation` finding carries `mad`, the mean
absolute deviation of the observed first-digit shares from
Benford's, with the chi-square statistic and the observed shares
per digit.


## What the detector does not report

- **Fixed-precision decimal columns.** Money recorded with cents
  is not digit-tested in this version: the last significant
  digit of such a column is the cents digit, while the tests are
  defined on integer-valued and float columns.
- **Benford-ineligible columns.** A column spanning fewer than
  three orders of magnitude, or with too few positive values, is
  not tested however its first digits look. The eligibility
  count is in the `digits` coverage record.
- **Heaping below the count floor.** Columns with fewer than
  500 values are not tested: at small counts the divisible share
  cannot be separated from chance at the stated bar.
